In [54]:
import shutil, gc
from pathlib import Path
BASE = Path("/content/tlab")

# удалить agent artifacts
for p in [BASE/"features"/"agent_features.parquet"]:
    if p.exists(): p.unlink()
shutil.rmtree(BASE/"features"/"agent_cands", ignore_errors=True)
(BASE/"features"/"agent_cands").mkdir(parents=True, exist_ok=True)

# проверить размеры
for d in ["features", "features/manual_weeks", "cands", "data", "artifacts"]:
    p = BASE / d
    if p.exists():
        files = [f for f in p.glob("*.parquet") if f.is_file()]
        total = sum(f.stat().st_size for f in files) / 1e6 if files else 0
        print(f"{d}/: {len(files)} files, {total:.1f} MB")

gc.collect()
import psutil
print(f"RAM: {psutil.virtual_memory().used/1e9:.1f} GB")

features/: 1 files, 37.2 MB
features/manual_weeks/: 16 files, 37.5 MB
cands/: 16 files, 5.0 MB
data/: 1 files, 14.2 MB
artifacts/: 2 files, 3.7 MB
RAM: 3.5 GB


In [55]:
!pip install -q catboost lightgbm polars huggingface_hub tqdm

In [56]:
import os, gc, json, random, shutil, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional
import numpy as np
import polars as pl
import pandas as pd
import psutil
from sklearn.metrics import roc_auc_score, log_loss
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")

DAY, WEEK, SEED = 86400, 604800, 42
N_USERS, HISTORY_DAYS, N_POP, MAX_WINDOW = 400, 14, 20, 60
random.seed(SEED); np.random.seed(SEED)

BASE = Path("/content/tlab")
for d in ("data","cands","features","features/manual_weeks",
          "features/agent_cands","artifacts","hf_cache"):
    (BASE/d).mkdir(parents=True, exist_ok=True)
HF_CACHE = BASE/"hf_cache"

In [57]:
def hf_get(name, force=False):
    return hf_hub_download("yandex/yambda", name, repo_type="dataset",
                           cache_dir=str(HF_CACHE), force_download=force)
def safe_write(df, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); df.write_parquet(path)
def safe_load_hf(name):
    try: return pl.read_parquet(hf_get(name))
    except Exception:
        return pl.read_parquet(hf_get(name, force=True))

In [58]:

Entity = Literal["user","item","user_item","user_artist","user_album"]
Agg    = Literal["count","sum","mean","std","max","min","nunique","ratio"]

@dataclass(frozen=True)
class FeatureSpec:
    name: str
    entity: Entity
    agg: Agg
    event_types: tuple = ("listen",)
    metric: Optional[str] = None
    window_days: Optional[int] = None
    filter_is_organic: Optional[int] = None
    filter_min_played_ratio: Optional[float] = None
    ratio_num: Optional["FeatureSpec"] = None
    ratio_den: Optional["FeatureSpec"] = None

class FeatureExecutor:
    def __init__(self, events, artist_map=None, album_map=None):
        self.events = events; self.artist_map = artist_map; self.album_map = album_map
    def _slice(self, spec):
        ev = self.events
        if spec.event_types:
            ev = ev.filter(pl.col("event_type").is_in(list(spec.event_types)))
        if spec.filter_is_organic is not None:
            ev = ev.filter(pl.col("is_organic") == spec.filter_is_organic)
        if spec.filter_min_played_ratio is not None:
            ev = ev.filter(pl.col("played_ratio_pct") >= spec.filter_min_played_ratio)
        return ev
    def _key(self, e):
        return {"user":["uid"],"item":["item_id"],"user_item":["uid","item_id"],
                "user_artist":["uid","artist_id"],"user_album":["uid","album_id"]}[e]
    def _agg(self, spec):
        if spec.agg == "count": return pl.len().alias(spec.name)
        if spec.agg == "nunique":
            col = "item_id" if spec.entity=="user" else "uid" if spec.entity=="item" else "item_id"
            return pl.col(col).n_unique().alias(spec.name)
        m = pl.col(spec.metric)
        return {"sum":m.sum(),"mean":m.mean(),"std":m.std(),"max":m.max(),"min":m.min()}[spec.agg].alias(spec.name)
    def compute(self, cands, spec):
        if spec.agg == "ratio":
            a = self.compute(cands, spec.ratio_num).rename({spec.ratio_num.name:"__n"})
            b = self.compute(cands, spec.ratio_den).rename({spec.ratio_den.name:"__d"})
            out = a.join(b, on=["uid","item_id","t"], how="left")
            return out.with_columns((pl.col("__n")/pl.col("__d").fill_null(1))
                .fill_nan(0).fill_null(0).cast(pl.Float32).alias(spec.name)) \
                .select("uid","item_id","t",spec.name)
        ev = self._slice(spec); cands_ = cands
        if spec.entity == "user_artist":
            ev = ev.join(self.artist_map, on="item_id", how="inner")
            cands_ = cands.join(self.artist_map, on="item_id", how="left")
        elif spec.entity == "user_album":
            ev = ev.join(self.album_map, on="item_id", how="inner")
            cands_ = cands.join(self.album_map, on="item_id", how="left")
        keys = self._key(spec.entity)
        left = cands_.select(keys + ["t"]).unique()
        merged = left.join(ev, on=keys, how="left")
        if spec.window_days is not None:
            merged = merged.filter((pl.col("timestamp") >= pl.col("t") - spec.window_days*DAY)
                                   & (pl.col("timestamp") < pl.col("t")))
        else:
            merged = merged.filter(pl.col("timestamp") < pl.col("t"))
        agg = merged.group_by(keys+["t"]).agg(self._agg(spec))
        fc = list(dict.fromkeys(keys + ["uid","item_id","t"]))
        return (cands_.select(fc).join(agg, on=keys+["t"], how="left")
                .with_columns(pl.col(spec.name).fill_null(0).fill_nan(0).cast(pl.Float32))
                .select("uid","item_id","t",spec.name))
    def compute_many(self, cands, specs):
        out = cands.select(["uid","item_id","t","y","split","source"])
        for s in specs:
            col = self.compute(cands, s).select(["uid","item_id","t",s.name])
            out = out.join(col, on=["uid","item_id","t"], how="left")
        return out.unique(subset=["uid","item_id","t"], keep="first")

In [59]:
MANUAL_SPECS = []; A = MANUAL_SPECS.append
for w in (1,7,30):
    A(FeatureSpec(f"user_listen_count_{w}d","user","count",("listen",),window_days=w))
    A(FeatureSpec(f"user_like_count_{w}d","user","count",("like",),window_days=w))
for w in (7,30):
    A(FeatureSpec(f"user_dislike_count_{w}d","user","count",("dislike",),window_days=w))
    A(FeatureSpec(f"user_uniq_items_{w}d","user","nunique",("listen",),window_days=w))
    A(FeatureSpec(f"user_mean_played_{w}d","user","mean",("listen",),metric="played_ratio_pct",window_days=w))
A(FeatureSpec("user_organic_listen_30d","user","count",("listen",),window_days=30,filter_is_organic=1))
for w in (1,7,30):
    A(FeatureSpec(f"item_listen_count_{w}d","item","count",("listen",),window_days=w))
for w in (7,30):
    A(FeatureSpec(f"item_like_count_{w}d","item","count",("like",),window_days=w))
    A(FeatureSpec(f"item_uniq_users_{w}d","item","nunique",("listen",),window_days=w))
A(FeatureSpec("item_dislike_count_30d","item","count",("dislike",),window_days=30))
A(FeatureSpec("item_mean_played_30d","item","mean",("listen",),metric="played_ratio_pct",window_days=30))
A(FeatureSpec("item_mean_length_30d","item","mean",("listen",),metric="track_length_seconds",window_days=30))
A(FeatureSpec("ui_listen_count_30d","user_item","count",("listen",),window_days=30))
A(FeatureSpec("ui_listen_count_60d","user_item","count",("listen",),window_days=60))
A(FeatureSpec("ui_like_count_30d","user_item","count",("like",),window_days=30))
A(FeatureSpec("ui_mean_played_30d","user_item","mean",("listen",),metric="played_ratio_pct",window_days=30))
A(FeatureSpec("ui_max_played_60d","user_item","max",("listen",),metric="played_ratio_pct",window_days=60))
A(FeatureSpec("user_artist_listen_count_30d","user_artist","count",("listen",),window_days=30))
A(FeatureSpec("user_artist_like_count_30d","user_artist","count",("like",),window_days=30))
A(FeatureSpec(name="user_ctr_7d",entity="user",agg="ratio",
    ratio_num=FeatureSpec("__n","user","count",("like",),window_days=7),
    ratio_den=FeatureSpec("__d","user","count",("listen",),window_days=7)))
A(FeatureSpec(name="user_ctr_30d",entity="user",agg="ratio",
    ratio_num=FeatureSpec("__n","user","count",("like",),window_days=30),
    ratio_den=FeatureSpec("__d","user","count",("listen",),window_days=30)))
A(FeatureSpec(name="item_like_rate_7d",entity="item",agg="ratio",
    ratio_num=FeatureSpec("__n","item","count",("like",),window_days=7),
    ratio_den=FeatureSpec("__d","item","count",("listen",),window_days=7)))

TRAIN_WEEKS = list(range(10,30,2)); VAL_WEEKS=[31,33,35]; TEST_WEEKS=[36,38,40]
ALL_WEEKS = TRAIN_WEEKS + VAL_WEEKS + TEST_WEEKS
split_of = {**{w:"train" for w in TRAIN_WEEKS}, **{w:"val" for w in VAL_WEEKS}, **{w:"test" for w in TEST_WEEKS}}

SUB_PATH  = BASE/"data"/f"sub_{N_USERS}.parquet"
FEAT_PATH = BASE/"features"/"manual.parquet"
CAND_DIR  = BASE/"cands"
AGENT_DIR = BASE/"features"/"agent_cands"
CAND_FEAT = BASE/"features"/"agent_features.parquet"
VAL_PRED  = BASE/"artifacts"/"val_pred.parquet"
TEST_PRED = BASE/"artifacts"/"test_pred.parquet"

In [60]:
if not SUB_PATH.exists():
    ev_raw = safe_load_hf("flat/50m/multi_event.parquet")
    users = ev_raw.select("uid").unique().sample(N_USERS, seed=SEED)
    sub = ev_raw.join(users, on="uid", how="semi")
    safe_write(sub, SUB_PATH); del ev_raw; gc.collect()
sub = pl.read_parquet(SUB_PATH).unique(subset=["uid","item_id","timestamp","event_type"], keep="first")

artist_raw = safe_load_hf("artist_item_mapping.parquet")
album_raw  = safe_load_hf("album_item_mapping.parquet")
artist = artist_raw.join(sub.select("item_id").unique(), on="item_id", how="semi") \
                    .unique(subset=["item_id"], keep="first")
album  = album_raw.join(sub.select("item_id").unique(), on="item_id", how="semi") \
                   .unique(subset=["item_id"], keep="first")

def positives_at(ev, t):
    ts = pl.col("timestamp")
    return ev.filter((ts >= t) & (ts < t + WEEK) & (
        ((pl.col("event_type")=="listen") & (pl.col("played_ratio_pct")>50))
        | (pl.col("event_type")=="like"))).select("uid","item_id").unique()
def make_snapshot(ev, t):
    ts = pl.col("timestamp")
    lis = ev.filter(pl.col("event_type")=="listen")
    hist = lis.filter((ts >= t - HISTORY_DAYS*DAY) & (ts < t)).select("uid","item_id").unique() \
        .with_columns(pl.lit(True).alias("in_hist"))
    pop = lis.filter((ts >= t - WEEK) & (ts < t)).group_by("item_id").len().sort("len", descending=True) \
        .head(N_POP).select("item_id")
    users_ = hist.select("uid").unique()
    pop_pairs = users_.join(pop, how="cross").select("uid","item_id").with_columns(pl.lit(True).alias("in_pop"))
    cand = hist.join(pop_pairs, on=["uid","item_id"], how="full", coalesce=True).with_columns(
        pl.when(pl.col("in_hist").is_not_null() & pl.col("in_pop").is_not_null()).then(pl.lit("both"))
        .when(pl.col("in_hist").is_not_null()).then(pl.lit("hist"))
        .otherwise(pl.lit("pop")).alias("source")).select("uid","item_id","source")
    pos = positives_at(ev, t).with_columns(pl.lit(1, dtype=pl.UInt8).alias("y"))
    return cand.join(pos, on=["uid","item_id"], how="left").with_columns(
        pl.col("y").fill_null(0), pl.lit(t, dtype=pl.UInt32).alias("t")).unique(subset=["uid","item_id","t"])

missing = [w for w in ALL_WEEKS if not (CAND_DIR/f"w{w}.parquet").exists()]
for w in missing:
    out = make_snapshot(sub, w*WEEK)
    safe_write(out.with_columns(pl.lit(split_of[w]).alias("split")), CAND_DIR/f"w{w}.parquet")

if not FEAT_PATH.exists():
    weeks_dir = BASE/"features"/"manual_weeks"
    for w in ALL_WEEKS:
        out_path = weeks_dir/f"w{w}.parquet"
        if out_path.exists(): continue
        t = w * WEEK
        ev_week = sub.filter((pl.col("timestamp") >= t-MAX_WINDOW*DAY) & (pl.col("timestamp") < t))
        cands_w = pl.read_parquet(CAND_DIR/f"w{w}.parquet")
        ex = FeatureExecutor(ev_week, artist, album)
        df = ex.compute_many(cands_w, MANUAL_SPECS)
        safe_write(df, out_path); del df, cands_w, ex, ev_week; gc.collect()
    pl.scan_parquet(str(weeks_dir/"w*.parquet")).sink_parquet(FEAT_PATH)

feats = pl.read_parquet(FEAT_PATH).unique(subset=["uid","item_id","t"], keep="first")
cols = [c for c in feats.columns if c not in {"uid","item_id","t","y","split","source"}]


Metrics

In [61]:
def _ndcg(g,k):
    g = g.sort_values("score",ascending=False).head(k); y = g["y"].values
    if y.sum()==0: return 0.0
    d = 1/np.log2(np.arange(2,len(y)+2)); dcg = (y*d).sum()
    ideal = np.sort(y)[::-1]; idcg = (ideal*d[:len(ideal)]).sum()
    return float(dcg/idcg) if idcg>0 else 0.0
def _recall(g,k):
    g = g.sort_values("score",ascending=False).head(k); n = g["y"].sum()
    return 0.0 if n==0 else float(g["y"].sum()/n)
def _map(g,k):
    g = g.sort_values("score",ascending=False).head(k); y = g["y"].values
    if y.sum()==0: return 0.0
    cum = np.cumsum(y); prec = cum/(np.arange(len(y))+1)
    return float((prec*y).sum()/y.sum())
def full_eval(df, tag=""):
    y = df["y"].to_numpy(); s = df["score"].to_numpy()
    res = {"auc":float(roc_auc_score(y,s)), "logloss":float(log_loss(y,np.clip(s,1e-6,1-1e-6)))}
    pdf = df.select(["uid","t","y","score"]).to_pandas(); pdf["y"] = pdf["y"].astype(int)
    g = pdf.groupby(["uid","t"],sort=False)
    for k in (3,10):
        res[f"ndcg@{k}"]   = float(g.apply(lambda x: _ndcg(x,k), include_groups=False).mean())
        res[f"recall@{k}"] = float(g.apply(lambda x: _recall(x,k), include_groups=False).mean())
        res[f"map@{k}"]    = float(g.apply(lambda x: _map(x,k), include_groups=False).mean())
    return {f"{tag}_{k}":v for k,v in res.items()} if tag else res
def _to_np(df, c):
    return df.select([pl.col(x).cast(pl.Float32).fill_null(0).fill_nan(0) for x in c]) \
             .to_numpy().astype(np.float32)

baseline

In [62]:
if VAL_PRED.exists() or TEST_PRED.exists():
    for p in (VAL_PRED, TEST_PRED):
        if p.exists(): p.unlink()

import catboost as cb
tr = feats.filter(pl.col("split")=="train")
va = feats.filter(pl.col("split")=="val")
te = feats.filter(pl.col("split")=="test")
Xtr = _to_np(tr, cols); ytr = tr["y"].to_numpy().astype(np.int8)
Xva = _to_np(va, cols); yva = va["y"].to_numpy().astype(np.int8)
model = cb.CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC", random_seed=SEED, verbose=False,
    early_stopping_rounds=100, task_type="CPU")
model.fit(Xtr, ytr, eval_set=(Xva, yva))
def _pred(df):
    X = _to_np(df, cols); s = model.predict_proba(X)[:,1].astype(np.float32)
    return df.select(["uid","item_id","t","y","source"]).with_columns(pl.Series("score",s))
val_pred = _pred(va); test_pred = _pred(te)
safe_write(val_pred, VAL_PRED); safe_write(test_pred, TEST_PRED)
model.save_model(str(BASE/"artifacts"/"catboost_baseline.cbm"))
del Xtr, Xva, tr, va, te, model; gc.collect()

199

candidates

In [63]:
existing = {s.name for s in MANUAL_SPECS}
CANDIDATE_SPECS = []
def M(s):
    if s.name not in existing:
        CANDIDATE_SPECS.append(s); existing.add(s.name)
for w in (3,14,60,90):
    M(FeatureSpec(f"ui_mean_played_{w}d","user_item","mean",("listen",),metric="played_ratio_pct",window_days=w))
    M(FeatureSpec(f"ui_std_played_{w}d","user_item","std",("listen",),metric="played_ratio_pct",window_days=w))
for w in (90,120):
    M(FeatureSpec(f"ui_listen_count_{w}d","user_item","count",("listen",),window_days=w))
for w in (3,14):
    M(FeatureSpec(f"user_listen_count_{w}d","user","count",("listen",),window_days=w))
    M(FeatureSpec(f"user_uniq_items_{w}d","user","nunique",("listen",),window_days=w))
for w in (3,14,60):
    M(FeatureSpec(f"user_mean_played_{w}d","user","mean",("listen",),metric="played_ratio_pct",window_days=w))
    M(FeatureSpec(f"user_std_played_{w}d","user","std",("listen",),metric="played_ratio_pct",window_days=w))
for w in (7,30):
    M(FeatureSpec(f"user_max_played_{w}d","user","max",("listen",),metric="played_ratio_pct",window_days=w))
    M(FeatureSpec(f"item_max_played_{w}d","item","max",("listen",),metric="played_ratio_pct",window_days=w))
    M(FeatureSpec(f"item_std_played_{w}d","item","std",("listen",),metric="played_ratio_pct",window_days=w))
    M(FeatureSpec(f"user_organic_count_{w}d","user","count",("listen",),window_days=w,filter_is_organic=1))
    M(FeatureSpec(f"user_organic_like_{w}d","user","count",("like",),window_days=w,filter_is_organic=1))
    M(FeatureSpec(f"ui_high_played_{w}d","user_item","count",("listen",),window_days=w,filter_min_played_ratio=80))
M(FeatureSpec(name="ui_personalization_7d",entity="user_item",agg="ratio",
    ratio_num=FeatureSpec("__n","user_item","count",("listen",),window_days=7),
    ratio_den=FeatureSpec("__d","user","count",("listen",),window_days=7)))
M(FeatureSpec(name="ui_engagement_share_30d",entity="user_item",agg="ratio",
    ratio_num=FeatureSpec("__n","user_item","count",("listen",),window_days=30),
    ratio_den=FeatureSpec("__d","user_item","count",("listen",),window_days=60)))
M(FeatureSpec(name="ui_like_rate_30d",entity="user_item",agg="ratio",
    ratio_num=FeatureSpec("__n","user_item","count",("like",),window_days=30),
    ratio_den=FeatureSpec("__d","user_item","count",("listen",),window_days=30)))
for w in (7,30):
    M(FeatureSpec(f"user_album_listen_count_{w}d","user_album","count",("listen",),window_days=w))
M(FeatureSpec("user_dislike_rate_30d","user","ratio",
    ratio_num=FeatureSpec("__n","user","count",("dislike",),window_days=30),
    ratio_den=FeatureSpec("__d","user","count",("listen",),window_days=30)))

agent features

In [64]:
if not CAND_FEAT.exists():
    for w in ALL_WEEKS:
        out_path = AGENT_DIR/f"w{w}.parquet"
        if out_path.exists():
            try: _ = pl.read_parquet(out_path, n_rows=1); continue
            except: out_path.unlink()
        t = w * WEEK
        ev_week = sub.filter((pl.col("timestamp") >= t-100*DAY) & (pl.col("timestamp") < t))
        cands_w = pl.read_parquet(CAND_DIR/f"w{w}.parquet")
        ex = FeatureExecutor(ev_week, artist, album)
        df = ex.compute_many(cands_w, CANDIDATE_SPECS).unique(subset=["uid","item_id","t"], keep="first")
        safe_write(df, out_path); del df, cands_w, ex, ev_week; gc.collect()
    pl.scan_parquet(str(AGENT_DIR/"w*.parquet")).sink_parquet(CAND_FEAT)

eval

In [65]:
import lightgbm as lgb
ST, SV = 100_000, 50_000
tr_b = feats.filter(pl.col("split")=="train"); va_b = feats.filter(pl.col("split")=="val")
tr_s = tr_b.sample(n=min(ST, tr_b.height), seed=SEED)
va_s = va_b.sample(n=min(SV, va_b.height), seed=SEED)
Xtr_base = _to_np(tr_s, cols); Xva_base = _to_np(va_s, cols)
ytr_s = tr_s["y"].to_numpy().astype(np.int8); yva_s = va_s["y"].to_numpy().astype(np.int8)
train_keys = tr_s.select(["uid","item_id","t"]); val_keys = va_s.select(["uid","item_id","t"])

def load_col(name):
    return (pl.scan_parquet(str(CAND_FEAT))
            .select(["uid","item_id","t", name])
            .unique(subset=["uid","item_id","t"], keep="first")
            .collect())
def quick_eval(Xtr, ytr, Xva, yva, n=100):
    m = lgb.LGBMClassifier(n_estimators=n, learning_rate=0.1, num_leaves=15,
        min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(Xtr, ytr); auc = float(roc_auc_score(yva, m.predict_proba(Xva)[:,1]))
    del m; gc.collect(); return auc

base_auc = quick_eval(Xtr_base, ytr_s, Xva_base, yva_s)
deltas = []
for spec in CANDIDATE_SPECS:
    try:
        col_df = load_col(spec.name)
        x_tr = train_keys.join(col_df, on=["uid","item_id","t"], how="left") \
                        .select(spec.name).to_numpy().reshape(-1,1).astype(np.float32)
        x_va = val_keys.join(col_df, on=["uid","item_id","t"], how="left") \
                        .select(spec.name).to_numpy().reshape(-1,1).astype(np.float32)
        Xtr = np.hstack([Xtr_base, x_tr]); Xva = np.hstack([Xva_base, x_va])
        auc = quick_eval(Xtr, ytr_s, Xva, yva_s)
        deltas.append((spec.name, auc, auc - base_auc))
        del x_tr, x_va, Xtr, Xva, col_df; gc.collect()
    except Exception:
        deltas.append((spec.name, 0.0, -999)); gc.collect()

delta_df = pl.DataFrame(deltas, schema=["feature","val_auc","delta"], orient="row").sort("delta", descending=True)

forward selection

In [66]:
selected = []; cur_auc = base_auc
THRESH, MAX_SEL = 0.0005, 10
Xtr_sel, Xva_sel = Xtr_base.copy(), Xva_base.copy()
for row in delta_df.filter(pl.col("delta") > 0).iter_rows(named=True):
    if len(selected) >= MAX_SEL: break
    name = row["feature"]
    col_df = load_col(name)
    x_tr = train_keys.join(col_df, on=["uid","item_id","t"], how="left") \
                    .select(name).to_numpy().reshape(-1,1).astype(np.float32)
    x_va = val_keys.join(col_df, on=["uid","item_id","t"], how="left") \
                    .select(name).to_numpy().reshape(-1,1).astype(np.float32)
    Xtr_try = np.hstack([Xtr_sel, x_tr]); Xva_try = np.hstack([Xva_sel, x_va])
    auc = quick_eval(Xtr_try, ytr_s, Xva_try, yva_s)
    if auc - cur_auc > THRESH:
        selected.append(name); Xtr_sel = Xtr_try; Xva_sel = Xva_try
        cur_auc = auc
    else:
        del Xtr_try, Xva_try, x_tr, x_va
    del col_df; gc.collect()

final model

In [67]:
FINAL_COLS = cols + selected
def build_split(s):
    df_base = feats.filter(pl.col("split")==s).unique(subset=["uid","item_id","t"], keep="first")
    if not selected: return df_base
    keys = df_base.select(["uid","item_id","t"]).unique(subset=["uid","item_id","t"])
    lf = (pl.scan_parquet(str(CAND_FEAT))
          .select(["uid","item_id","t"] + selected)
          .unique(subset=["uid","item_id","t"], keep="first"))
    df_new = keys.join(lf.collect(), on=["uid","item_id","t"], how="left")
    df_new = df_new.with_columns([pl.col(c).fill_null(0).fill_nan(0).cast(pl.Float32) for c in selected])
    return df_base.join(df_new, on=["uid","item_id","t"], how="left")

tr_f = build_split("train"); va_f = build_split("val"); te_f = build_split("test")
Xtr_f = _to_np(tr_f, FINAL_COLS); ytr_f = tr_f["y"].to_numpy().astype(np.int8)
Xva_f = _to_np(va_f, FINAL_COLS); yva_f = va_f["y"].to_numpy().astype(np.int8)
final_model = cb.CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC", random_seed=SEED, verbose=False,
    early_stopping_rounds=100, task_type="CPU")
final_model.fit(Xtr_f, ytr_f, eval_set=(Xva_f, yva_f))
def pred_final(df):
    X = _to_np(df, FINAL_COLS); s = final_model.predict_proba(X)[:,1].astype(np.float32)
    return df.select(["uid","item_id","t","y","source"]).with_columns(pl.Series("score", s))
val_pred_f = pred_final(va_f); test_pred_f = pred_final(te_f)

b = full_eval(test_pred); a = full_eval(test_pred_f)
results = {"selected_features": selected, "baseline_test": b, "agent_test": a,
           "delta_auc": a["auc"]-b["auc"], "delta_ndcg10": a["ndcg@10"]-b["ndcg@10"],
           "baseline_lgbm": float(base_auc), "final_lgbm": float(cur_auc)}
with open(BASE/"artifacts"/"agent_results.json","w") as f:
    json.dump(results, f, indent=2)

вывод

In [68]:
print("=" * 52)
print("T-LAB RESULT (test)")
print("=" * 52)
print(f"{'metric':<14} {'baseline':>10} {'agent':>10} {'delta':>10}")
print("-" * 52)
for m in ("auc","logloss","ndcg@3","ndcg@10","recall@10","map@10"):
    print(f"{m:<14} {b[m]:>10.4f} {a[m]:>10.4f} {a[m]-b[m]:>+10.4f}")
print("-" * 52)
print(f"\nselected features ({len(selected)}):")
for f in selected:
    print(f"  + {f}")
print(f"\ndelta_auc      = {results['delta_auc']:+.4f}")
print(f"delta_ndcg@10  = {results['delta_ndcg10']:+.4f}")

T-LAB RESULT (test)
metric           baseline      agent      delta
----------------------------------------------------
auc                0.7968     0.8050    +0.0082
logloss            0.3128     0.3089    -0.0039
ndcg@3             0.5277     0.5552    +0.0275
ndcg@10            0.5543     0.5707    +0.0165
recall@10          0.6819     0.6830    +0.0011
map@10             0.4844     0.5039    +0.0195
----------------------------------------------------

selected features (2):
  + ui_personalization_7d
  + ui_high_played_7d

delta_auc      = +0.0082
delta_ndcg@10  = +0.0165
